# Confiabilidade dos indicadores

Compara o resumo do CPAP, os eventos lidos no fluxo de 25 Hz e o Garmin. Não há polissonografia com o aparelho, então a confiança de cada indicador sai de cinco critérios: repetibilidade, resposta à pressão, convergência com o relógio, independência de vazamento e evidência publicada.

A queda de 30% do fluxo conta mais eventos que o AHI. O teste justo usa o percentual do guia da ResMed (hipopneia até 50%, apneia acima de 75%) sem somar os dois: o detector de 50% já inclui as quedas mais fundas.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

from sleep_analysis.db import connect
from sleep_analysis.features import build_half_counts, build_night_features
from sleep_analysis.paths import PROCESSED_DIR, PROJECT_DIR
from sleep_analysis.reliability import (
    holdout_validation,
    load_indicator_config,
    quality_index,
    score_indicators,
    select_core,
)
from sleep_analysis.undercount import (
    DEVICE_RESPIRATORY,
    bland_altman,
    discordant_breakdown,
    event_agreement,
    lin_ccc,
    summarize_undercount,
    threshold_curve,
)

REPORTS = PROJECT_DIR / "data" / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)
config = load_indicator_config()
connection = connect()
features = build_night_features(connection)
halves = build_half_counts(connection)
features.to_parquet(PROCESSED_DIR / "night_features.parquet", index=False)
comparable = features[features["usage_hours"] >= config["usage_min_hours"]]
print(f"{len(features)} noites no resumo; {len(comparable)} com uso ≥ {config['usage_min_hours']:.0f} h; {int(features['has_flow'].sum())} com fluxo")


## O que a literatura sustenta

Nota A: medida direta ou critério consagrado. Nota B: definição publicada, com viés ou imprecisão conhecidos. Nota C: proxy deste projeto ou índice proprietário sem validação encontrada. Número sem fonte não entra.


In [ ]:
evidence = pd.DataFrame(config["indicators"])[["name", "dimension", "evidence", "source"]]
evidence = evidence.rename(columns={"name": "indicador", "dimension": "dimensão", "evidence": "nota"})
evidence


## O aparelho conta menos eventos?

A curva é a mediana de eventos por hora de uso. O AHI do aparelho entra como referência. Sensibilidade: a marca da ResMed cai dentro do evento do fluxo, com 15 s de folga. Valor preditivo: o inverso.


In [ ]:
curve = threshold_curve(features, config["usage_min_hours"])
curve.to_csv(REPORTS / "07_threshold_curve.csv", index=False)
fig, ax = plt.subplots(figsize=(7, 4))
flow = curve.dropna(subset=["drop_fraction"])
ax.plot(flow["drop_fraction"] * 100, flow["median_per_h"], marker="o", label="fluxo")
device_median = curve.loc[curve["source"] == "ahi", "median_per_h"].iloc[0]
ax.axhline(device_median, color="0.4", linestyle="--", label="AHI do aparelho")
ax.set_xlabel("Queda mínima da ventilação (%)")
ax.set_ylabel("Mediana, eventos por hora de uso")
ax.set_title("Quanto o limiar muda a contagem")
ax.legend()
fig.tight_layout()
fig.savefig(REPORTS / "07_threshold_curve.png", dpi=120)
curve.round(3)


In [ ]:
device = connection.execute("select night, timestamp, event_type, duration_s from events").fetchdf()
own = connection.execute("select night, timestamp, event_type, duration_s from respiratory_events").fetchdf()
rows = []
for label, device_types, own_type in [
    ("aparelho → queda 30%", DEVICE_RESPIRATORY, "hypopnea30"),
    ("aparelho → queda 50%", DEVICE_RESPIRATORY, "hypopnea50"),
    ("apneia → queda 75%", ["obstructive_apnea", "central_apnea", "unclassified_apnea"], "apnea75"),
    ("apneia → queda 90%", ["obstructive_apnea", "central_apnea", "unclassified_apnea"], "apnea90"),
    ("RERA → surge", ["arousal"], "surge"),
    ("RERA → rera_like", ["arousal"], "rera_like"),
]:
    agreement = event_agreement(device, own, device_types, own_type)
    rows.append({
        "comparação": label,
        "eventos aparelho": agreement["device_events"],
        "eventos fluxo": agreement["own_events"],
        "sensibilidade": agreement["sensitivity"],
        "valor preditivo": agreement["ppv"],
    })
agreement_table = pd.DataFrame(rows)
agreement_table.to_csv(REPORTS / "07_event_agreement.csv", index=False)
agreement_table.round(3)


In [ ]:
signals = connection.execute("select night, timestamp, leak from signals_2s").fetchdf()
sessions = connection.execute("select night, session_start from sessions").fetchdf()
settings = connection.execute("select night, ramp_enabled, ramp_time_min from nights").fetchdf()
discordant = discordant_breakdown(
    device, own, signals, sessions, settings,
    own[own["event_type"] == "surge"], own[own["event_type"] == "wake_like"],
)
discordant.to_csv(REPORTS / "07_discordant.csv", index=False)
discordant.round(3)


In [ ]:
paired = comparable.dropna(subset=["ahi", "hypopnea50_per_usage_h", "hypopnea30_per_usage_h"])
altman = bland_altman(paired["ahi"], paired["hypopnea50_per_usage_h"])
ccc = lin_ccc(paired["ahi"], paired["hypopnea50_per_usage_h"])
rate = stats.spearmanr(paired["device_resp_rate"], paired["flow_resp_rate"], nan_policy="omit").statistic
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(altman["level"], altman["difference"], s=12, alpha=0.7)
ax.axhline(altman["bias"], color="C0", label=f"viés {altman['bias']:.2f}")
ax.axhline(altman["loa_low"], color="0.5", linestyle="--")
ax.axhline(altman["loa_high"], color="0.5", linestyle="--", label="limites de concordância")
ax.set_xlabel("Média dos dois índices (/h)")
ax.set_ylabel("Fluxo a 50% − AHI (/h)")
ax.set_title("Bland-Altman, mesma hora de uso")
ax.legend()
fig.tight_layout()
fig.savefig(REPORTS / "07_bland_altman.png", dpi=120)
unexplained = discordant.loc[discordant["category"] == "unexplained", "own_pct"].iloc[0]
flow50 = curve.loc[curve["source"] == "hypopnea50_per_usage_h", "median_per_h"].iloc[0]
flow30 = curve.loc[curve["source"] == "hypopnea30_per_usage_h", "median_per_h"].iloc[0]
matched = agreement_table.loc[agreement_table["comparação"] == "aparelho → queda 50%"].iloc[0]
print(summarize_undercount(
    device_median, flow50, flow30, matched["sensibilidade"], matched["valor preditivo"],
    unexplained, altman["bias"], ccc,
))
print(f"Frequência do aparelho e do fluxo: Spearman {rate:.2f}. CCC do AHI com a queda de 50%: {ccc:.2f}.")
print(f"O viés cresce com o nível da noite (Spearman {altman['bias_vs_level']:.2f}).")


## Nota de cada indicador

Repetibilidade junta o ICC de noites vizinhas na mesma pressão e a confiabilidade das metades (horas pares contra ímpares, corrigida por Spearman-Brown). Resposta à pressão usa só as noites coladas a uma troca, para não misturar com a época. Convergência é com HRV, FC de repouso, despertares e estresse do Garmin, em desvio da mediana móvel de 14 noites. Indicador do Garmin não ganha ponto por concordar com outro campo do Garmin.

O corte para entrar no índice é ICC ≥ 0,40 ou confiabilidade das metades ≥ 0,70. ICC baixo, sozinho, não condena um indicador que deve mudar de uma noite para a outra; a metade da mesma noite é que diz se a medida é estável.


In [ ]:
scored = score_indicators(features, halves, config)
components = scored["components"]
show = [
    "indicator", "dimension", "evidence", "icc", "split_half", "repeatability", "responsiveness",
    "convergence", "artifact_independence", "coverage", "score", "effect_per_cm", "n_switches", "mdc95",
]
components[show].to_csv(REPORTS / "07_scorecard.csv", index=False)
print(f"Perturbar um peso em 0,1 muda a nota com Spearman mínimo de {scored['sensitivity']['score_spearman'].min():.2f}.")
components[show].round(3)


In [ ]:
order = components.sort_values("score")
fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(order["indicator"], order["score"], color="C0")
ax.set_xlabel("Nota (0 a 1)")
ax.set_title("Nota composta, pesos do config")
fig.tight_layout()
fig.savefig(REPORTS / "07_scores.png", dpi=120)
clusters = scored["clusters"]
redundant = clusters.groupby(clusters).apply(lambda group: list(group.index), include_groups=False)
redundant[redundant.str.len() > 1]


## Núcleo e índice

Entra no índice quem passa o corte de repetibilidade. Quem fica abaixo aparece na tabela e não compõe a nota da noite. O índice v1 é o do notebook 03 (surto, hipopneia de 30% mais RERA-like, estabilidade), com maior = pior, calibrado nas noites a 7 cmH2O antes de maio/2026. A validação usa só as noites de maio/2026 em diante, contra o Garmin.


In [ ]:
core = select_core(components, features, config)
core.to_csv(REPORTS / "07_core.csv", index=False)
passing = core[~core["below_icc"]].copy()
indexed = quality_index(features, passing, config)
validation = holdout_validation(indexed, features, config["anchors"], config["calibration_until"])
validation.to_csv(REPORTS / "07_holdout.csv", index=False)
print("Núcleo")
display_core = core[["indicator", "dimension", "role", "below_icc", "score", "icc", "split_half"]]
print(display_core.round(3).to_string(index=False))
print("\nValidação depois de", config["calibration_until"])
print(validation.round(3).to_string(index=False))
flow = validation[validation["index"] == "quality_v2_flow"]
v1 = validation[validation["index"] == "quality_v1"]
if not flow.empty and not v1.empty:
    print(
        f"No holdout, o índice do fluxo alinha em média {flow['aligned'].mean():.2f} com o Garmin; "
        f"o índice v1 alinha {v1['aligned'].mean():.2f}. Positivo é o sentido esperado."
    )
